# Logistic Regression Baseline Model

Train a logistic regression model as a baseline for comparison with the deep learning model.

## Model Details:
- Algorithm: Logistic Regression with L2 regularization
- Hyperparameter tuning: GridSearchCV with 5-fold cross-validation
- Regularization strengths tested: C ∈ {0.01, 0.1, 1, 10, 100}
- Solver: lbfgs (handles L2 penalty efficiently)

In [7]:
import os
import json
import pandas as pd
import numpy as np
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    classification_report,
    confusion_matrix
)

# Configuration
DATA_DIR = "../data"
MODELS_DIR = "../models"
RESULTS_DIR = "../results"

TRAIN_FILE = os.path.join(DATA_DIR, "train.csv")
VAL_FILE = os.path.join(DATA_DIR, "val.csv")
TEST_FILE = os.path.join(DATA_DIR, "test.csv")

MODEL_FILE = os.path.join(MODELS_DIR, "logistic_model.pkl")
PREDICTIONS_FILE = os.path.join(RESULTS_DIR, "logistic_predictions.csv")
METRICS_FILE = os.path.join(RESULTS_DIR, "logistic_metrics.json")

TARGET_COL = "status"
RANDOM_STATE = 42

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print("✓ Configuration loaded")

✓ Configuration loaded


## 1. Load Data

In [8]:
# Load data
train_df = pd.read_csv(TRAIN_FILE)
val_df = pd.read_csv(VAL_FILE)
test_df = pd.read_csv(TEST_FILE)

X_train = train_df.drop(columns=[TARGET_COL]).values
y_train = train_df[TARGET_COL].values

X_val = val_df.drop(columns=[TARGET_COL]).values
y_val = val_df[TARGET_COL].values

X_test = test_df.drop(columns=[TARGET_COL]).values
y_test = test_df[TARGET_COL].values

print(f"Training set: {X_train.shape}")
print(f"Validation set: {X_val.shape}")
print(f"Test set: {X_test.shape}")
print(f"\nClass distribution (train):")
print(f"  No default (0): {(y_train == 0).sum()} ({(y_train == 0).mean():.1%})")
print(f"  Default (1): {(y_train == 1).sum()} ({(y_train == 1).mean():.1%})")

Training set: (179250, 67)
Validation set: (14867, 67)
Test set: (14867, 67)

Class distribution (train):
  No default (0): 89625 (50.0%)
  Default (1): 89625 (50.0%)


## 2. Hyperparameter Tuning with GridSearchCV

Use 5-fold cross-validation to find the best regularization strength.

In [9]:
# Define parameter grid
param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],  # Regularization strength (inverse)
    'penalty': ['l2'],
    'solver': ['lbfgs'],
    'max_iter': [1000],
    'random_state': [RANDOM_STATE]
}

# Initialize base model
base_model = LogisticRegression()

# GridSearchCV with 5-fold cross-validation
grid_search = GridSearchCV(
    base_model,
    param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=2
)

print("Starting GridSearchCV...")
print(f"Testing {len(param_grid['C'])} values of C")
print("This may take a few minutes...\n")

grid_search.fit(X_train, y_train)

print("\n" + "=" * 70)
print("GRID SEARCH RESULTS")
print("=" * 70)
print(f"\nBest parameters: {grid_search.best_params_}")
print(f"Best cross-validation AUC-ROC: {grid_search.best_score_:.4f}")

# Get best model
model = grid_search.best_estimator_

print("\n✓ Model trained successfully")

Starting GridSearchCV...
Testing 5 values of C
This may take a few minutes...

Fitting 5 folds for each of 5 candidates, totalling 25 fits
[CV] END C=0.01, max_iter=1000, penalty=l2, random_state=42, solver=lbfgs; total time=   7.6s
[CV] END C=0.01, max_iter=1000, penalty=l2, random_state=42, solver=lbfgs; total time=   7.9s
[CV] END C=0.01, max_iter=1000, penalty=l2, random_state=42, solver=lbfgs; total time=   8.2s
[CV] END C=0.01, max_iter=1000, penalty=l2, random_state=42, solver=lbfgs; total time=   8.3s
[CV] END C=0.01, max_iter=1000, penalty=l2, random_state=42, solver=lbfgs; total time=   9.3s
[CV] END C=0.1, max_iter=1000, penalty=l2, random_state=42, solver=lbfgs; total time=  14.2s
[CV] END C=0.1, max_iter=1000, penalty=l2, random_state=42, solver=lbfgs; total time=  15.6s
[CV] END C=0.1, max_iter=1000, penalty=l2, random_state=42, solver=lbfgs; total time=  17.0s
[CV] END C=0.1, max_iter=1000, penalty=l2, random_state=42, solver=lbfgs; total time=  17.8s
[CV] END C=1, max_i

## 3. Evaluate on Validation Set

In [10]:
# Predictions on validation set
y_val_proba = model.predict_proba(X_val)[:, 1]
y_val_pred = model.predict(X_val)

# Calculate metrics
val_auc_roc = roc_auc_score(y_val, y_val_proba)
val_auc_pr = average_precision_score(y_val, y_val_proba)
val_brier = brier_score_loss(y_val, y_val_proba)

print("=" * 70)
print("VALIDATION SET PERFORMANCE")
print("=" * 70)
print(f"\nMetrics:")
print(f"  AUC-ROC: {val_auc_roc:.4f}")
print(f"  AUC-PR: {val_auc_pr:.4f}")
print(f"  Brier Score: {val_brier:.4f}")

print("\nClassification Report:")
print(classification_report(y_val, y_val_pred, digits=4))

print("Confusion Matrix:")
cm = confusion_matrix(y_val, y_val_pred)
print(cm)
print(f"\nTrue Negatives: {cm[0,0]}, False Positives: {cm[0,1]}")
print(f"False Negatives: {cm[1,0]}, True Positives: {cm[1,1]}")
print("=" * 70)

VALIDATION SET PERFORMANCE

Metrics:
  AUC-ROC: 0.8418
  AUC-PR: 0.7709
  Brier Score: 0.1338

Classification Report:
              precision    recall  f1-score   support

           0     0.8878    0.8895    0.8887     11203
           1     0.6602    0.6564    0.6583      3664

    accuracy                         0.8320     14867
   macro avg     0.7740    0.7729    0.7735     14867
weighted avg     0.8317    0.8320    0.8319     14867

Confusion Matrix:
[[9965 1238]
 [1259 2405]]

True Negatives: 9965, False Positives: 1238
False Negatives: 1259, True Positives: 2405


## 4. Evaluate on Test Set

In [11]:
# Predictions on test set
y_test_proba = model.predict_proba(X_test)[:, 1]
y_test_pred = model.predict(X_test)

# Calculate metrics
test_auc_roc = roc_auc_score(y_test, y_test_proba)
test_auc_pr = average_precision_score(y_test, y_test_proba)
test_brier = brier_score_loss(y_test, y_test_proba)

print("=" * 70)
print("TEST SET PERFORMANCE")
print("=" * 70)
print(f"\nMetrics:")
print(f"  AUC-ROC: {test_auc_roc:.4f}")
print(f"  AUC-PR: {test_auc_pr:.4f}")
print(f"  Brier Score: {test_brier:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_test_pred, digits=4))

print("Confusion Matrix:")
cm = confusion_matrix(y_test, y_test_pred)
print(cm)
print(f"\nTrue Negatives: {cm[0,0]}, False Positives: {cm[0,1]}")
print(f"False Negatives: {cm[1,0]}, True Positives: {cm[1,1]}")
print("=" * 70)

TEST SET PERFORMANCE

Metrics:
  AUC-ROC: 0.8512
  AUC-PR: 0.7800
  Brier Score: 0.1312

Classification Report:
              precision    recall  f1-score   support

           0     0.8911    0.8913    0.8912     11203
           1     0.6674    0.6670    0.6672      3664

    accuracy                         0.8360     14867
   macro avg     0.7793    0.7792    0.7792     14867
weighted avg     0.8360    0.8360    0.8360     14867

Confusion Matrix:
[[9985 1218]
 [1220 2444]]

True Negatives: 9985, False Positives: 1218
False Negatives: 1220, True Positives: 2444


## 5. Save Model and Results

In [12]:
# Save model
joblib.dump(model, MODEL_FILE)
print(f"✓ Model saved to {MODEL_FILE}")

# Save predictions
predictions_df = pd.DataFrame({
    'true_label': y_val,
    'predicted_probability': y_val_proba,
    'predicted_label': y_val_pred,
    'dataset': 'validation'
})
predictions_df.to_csv(PREDICTIONS_FILE, index=False)
print(f"✓ Predictions saved to {PREDICTIONS_FILE}")

# Save metrics
metrics = {
    'best_params': grid_search.best_params_,
    'best_cv_score': float(grid_search.best_score_),
    'validation_metrics': {
        'auc_roc': float(val_auc_roc),
        'auc_pr': float(val_auc_pr),
        'brier_score': float(val_brier),
        'dataset': 'Validation'
    },
    'test_metrics': {
        'auc_roc': float(test_auc_roc),
        'auc_pr': float(test_auc_pr),
        'brier_score': float(test_brier),
        'dataset': 'Test'
    }
}

with open(METRICS_FILE, 'w') as f:
    json.dump(metrics, f, indent=4)
print(f"✓ Metrics saved to {METRICS_FILE}")

✓ Model saved to ../models/logistic_model.pkl
✓ Predictions saved to ../results/logistic_predictions.csv
✓ Metrics saved to ../results/logistic_metrics.json


## 6. Model Coefficients (Feature Importance)

Examine the learned coefficients to understand feature importance.

In [13]:
# Load feature names
feature_names_file = os.path.join(MODELS_DIR, "feature_names.txt")
with open(feature_names_file, 'r') as f:
    feature_names = [line.strip() for line in f if line.strip()]

# Get coefficients
coefficients = model.coef_[0]

# Create dataframe
coef_df = pd.DataFrame({
    'feature': feature_names,
    'coefficient': coefficients,
    'abs_coefficient': np.abs(coefficients)
})

# Sort by absolute value
coef_df = coef_df.sort_values('abs_coefficient', ascending=False)

print("=" * 70)
print("TOP 10 MOST IMPORTANT FEATURES")
print("=" * 70)
print("\nPositive coefficients increase default risk")
print("Negative coefficients decrease default risk\n")

print(coef_df[['feature', 'coefficient']].head(10).to_string(index=False))

# Save top features for use in other notebooks
top_features_file = os.path.join(RESULTS_DIR, "top_features.csv")
coef_df['effect'] = coef_df['coefficient'].apply(lambda x: 'increases_risk' if x >= 0 else 'decreases_risk')
coef_df['odds_ratio'] = np.exp(coef_df['coefficient'])
coef_df.to_csv(top_features_file, index=False)
print(f"\n✓ Feature importance saved to {top_features_file}")

TOP 10 MOST IMPORTANT FEATURES

Positive coefficients increase default risk
Negative coefficients decrease default risk

               feature  coefficient
      credit_type_EQUI    37.671285
       credit_type_EXP   -11.701564
       credit_type_CIB   -11.653939
      credit_type_CRIF   -11.614870
  construction_type_mh     5.310043
security_type_Indriect     5.310043
       secured_by_land     5.310043
 lump_sum_payment_lpsm     2.619650
       secured_by_home    -2.609131
  construction_type_sb    -2.609131

✓ Feature importance saved to ../results/top_features.csv


## Summary

The logistic regression model serves as a strong baseline:
- Simple, interpretable linear model
- Coefficients directly show feature importance
- Fast to train and predict
- Performance will be compared with the MLP in `model_evaluation.ipynb`